**General Description**

The following notebook contains the code to create, train, validate, and test a rainfall-runoff model using the LSTM-MDN network architecture. The notebook support running experiments in different large-sample hydrology datasets including: CAMELS-GB, CAMELS-US, CAMELS-DE. The details for each dataset can be read from a .yml file.

***Authors:***
- Manuel Alvarez Chaves (manuel.alvarez-chaves@uwaterloo.c)
- Eduardo Acuña Espinoza (eduardo.espinoza@kit.edu)

In [ ]:
import datetime
import random
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr

from hy2dl.datasetzoo import get_dataset
from hy2dl.evaluation import get_tester
from hy2dl.evaluation.metrics import calculate_metrics
from hy2dl.modelzoo import get_model
from hy2dl.training.basetrainer import BaseTrainer
from hy2dl.utils.config import Config

base_dir = Path.cwd().resolve()
color_palette = {"observed": "#377eb8", "simulated": "#4daf4a"}

Part 1. Initialize information

In [ ]:
# Path to .yml file where the experiment settings are stored.
path_experiment_settings = "../examples/mdn.yml"

# Read experiment settings
config = Config(path_experiment_settings, base_dir=base_dir)
config.init_experiment()
config.dump()

Dataset = get_dataset(config)
Tester = get_tester(config)

Part 2. Create datasets and dataloaders used to train/validate the model

In [ ]:
# Create training dataset
training_dataset = Dataset(cfg=config, time_period="training")
training_dataset.setup_dataset()
# Initialize training object
trainer = BaseTrainer(cfg=config, training_dataset=training_dataset)

In [ ]:
validation_dataset = Dataset(cfg=config, time_period="validation")
validation_dataset.setup_dataset(check_nan=False, path_scaler = config.path_save_folder / "scaler.yml" )
tester_validation = Tester(cfg=config, evaluation_dataset=validation_dataset)

Part 3. Train model

In [ ]:
# Training report structure
config.logger.info("Training model".center(60, "-"))
config.logger.info(f"{'':^16}|{'Trainining':^21}|{'Validation':^21}|")
config.logger.info(f"{'Epoch':^5}|{'LR':^10}|{'Loss':^10}|{'Time':^10}|{'Metric':^10}|{'Time':^10}|")

# Loop through epochs
total_time = time.time()
for epoch in range(1, config.epochs + 1):
    trainer.train_model(epoch=epoch) # Training
    tester_validation.validate_model(model=trainer.model, epoch=epoch) # Validation
    config.logger.info(trainer.report + tester_validation.validation_report) #report
    
config.logger.info(f"Total training time: {datetime.timedelta(seconds=int(time.time() - total_time))}\n")
shutil.rmtree(tester_validation.path_zarr, ignore_errors=True) # delete validation results

Part 4. Test model

In [ ]:
# If I already trained a model, I can re-construct it using the saved parameters from a given epoch
#model = get_model(config).to(config.device)
#model.load_state_dict(torch.load(config.path_save_folder / "model" / f"model_epoch_{config.epochs}", map_location=config.device))

In [ ]:
testing_dataset = Dataset(cfg=config, time_period="testing")
testing_dataset.setup_dataset(check_nan=False, path_scaler = config.path_save_folder / "scaler.yml" )
tester_testing = Tester(cfg=config, evaluation_dataset=testing_dataset)
tester_testing.evaluate_model(model = trainer.model)

Part 5. Initial analysis

In [ ]:
test_results = xr.open_zarr(tester_testing.path_zarr)
testing_metrics = calculate_metrics(ds_results=test_results, metrics = config.testing_metrics)
testing_metrics.to_zarr(config.path_save_folder / "testing_metrics.zarr", mode="w")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))

ax.plot(data["date"], data["y_obs"][:, 0, 0], label="Observed", color=color_palette["observed"])
ax.plot(data["date"], data["y_hat_samples"].median(dim="num_samples")[:, 0, 0], label="Median", color=color_palette["simulated"])
ax.fill_between(data["date"], data["y_hat_samples"].quantile(0.05, dim="num_samples")[:, 0, 0], data["y_hat_samples"].quantile(0.95, dim="num_samples")[:, 0, 0], color=color_palette["simulated"], alpha=0.35, label="Range [0.05, 0.95]")
ax.grid(ls="--", alpha=0.5)

# Format plot
ax.legend()
ax.set_ylabel("Streamflow [mm/day]")
ax.set_xlabel("Date")
ax.set_title("Predictions from samples")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))

ax.plot(data["date"], data["y_obs"][:, 0, 0], label="Observed", color=color_palette["observed"])
ax.plot(data["date"], data["y_hat_mean"][:, 0, 0], label="Mean", color="k")
ax.plot(data["date"], data["y_hat_quantile"][:, 0, 3], label="Median", color=color_palette["simulated"])
ax.fill_between(data["date"], data["y_hat_quantile"][:, 0, 1, 0], data["y_hat_quantile"][:, 0, 5, 0], color=color_palette["simulated"], alpha=0.35, label="Range [0.05, 0.95]")
ax.grid(ls="--", alpha=0.5)

# Format plot
ax.legend()
ax.set_ylabel("Streamflow [mm/day]")
ax.set_xlabel("Date")
ax.set_title("Predictions from quantiles")
ax.set_ylim(-1, 15)
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=2, figsize=(10, 6), sharex=True)
axes[0].plot(data["date"], data["y_obs"][:, 0, 0], label="Observed", color=color_palette["observed"])
axes[0].plot(data["date"], data["y_obs"][:, 0, 0], label="Observed", color=color_palette["observed"])
axes[0].plot(data["date"], data["y_hat_samples"].median(dim="num_samples")[:, 0, 0], label="Median", color=color_palette["simulated"])
axes[0].fill_between(data["date"], data["y_hat_samples"].quantile(0.05, dim="num_samples")[:, 0, 0], data["y_hat_samples"].quantile(0.95, dim="num_samples")[:, 0, 0], color=color_palette["simulated"], alpha=0.35, label="Range [0.05, 0.95]")
axes[0].set_ylabel("Streamflow [mm/day]")

axes[1].plot(data["date"], data["weights"][:, 0, :, 0], label=[f"Comp. {i+1}" for i in range(data["weights"].shape[2])])
axes[1].set_ylabel("Mixture Weights")
axes[1].set_xlabel("Date")

# Format plot
for ax in axes:
    ax.grid(ls="--", alpha=0.5)
    ax.legend()

plt.show()